In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document
sample_doc = Document(
    page_content = "Hii i am Rohan",
    metadata = {"source" : "rohan is introducting himself"}
)

In [3]:
sample_doc

Document(metadata={'source': 'rohan is introducting himself'}, page_content='Hii i am Rohan')

In [4]:
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("Python.txt" , encoding = "utf-8")
document = loader.load()

C:\Users\Rohan\AppData\Local\Temp\ipykernel_8608\1149119972.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
C:\Users\Rohan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
document

[Document(metadata={'source': 'Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and m

In [1]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("pdfs/research2.pdf")
documentLoad = pdf_loader.load()
# documentLoad

C:\Users\Rohan\AppData\Local\Temp\ipykernel_12428\390075328.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader
C:\Users\Rohan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Ingestion Pipeline

### Document

In [2]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [3]:
def load_all_pdfs():
    path_dir = "pdfs"
    all_doc = []
    num_docs = 0
    for fileName in os.listdir(path_dir):
        if fileName.lower().endswith(".pdf"):
            #complete-file-path
            path_file = os.path.join(path_dir , fileName)
            loader = PyPDFLoader(path_file)
            document = loader.load()
            all_doc.extend(document)
            num_docs += 1
    print(f"number of docs : {num_docs}")
    print(f"pages of docs : {len(all_doc)}")
    return all_doc

all_doc_list = load_all_pdfs()

number of docs : 1
pages of docs : 21


In [4]:
type(all_doc_list)

list

### Chunks

In [19]:
#!pip install langchain_text_splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_chunks(documents , chunk_size = 500 , chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs
chunks = split_chunks(all_doc_list)
len(chunks)

244

### Embedding

In [6]:
from sentence_transformers import SentenceTransformer

In [7]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [8]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1497.52it/s]


embedding dimensions= 384


C:\Users\Rohan\AppData\Local\Temp\ipykernel_12428\4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Vector Store

In [9]:
import chromadb
import uuid

In [10]:
class VectorStoreManager:
    def __init__(self , persist_directory="vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory , exist_ok = True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self , documents , embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [11]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 244


In [12]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches: 100%|██████████| 8/8 [00:18<00:00,  2.26s/it]


embeddings shape: (244, 384)
total documents added in vector store= 244
docs in collection: 488


## Retrieval Pipeline

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [15]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [18]:
rag_retriever.retrieve("What is Large Language Models")

Batches: 100%|██████████| 1/1 [00:00<00:00, 55.50it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_9c04937c-3eb9-467a-9dda-1acf4fb4c571',
  'document': 'REFERENCES\n[1] N. Kandpal, H. Deng, A. Roberts, E. Wallace, and C. Raffel, “Large\nlanguage models struggle to learn long-tail knowledge,” in Interna-\ntional Conference on Machine Learning . PMLR, 2023, pp. 15 696–\n15 707.\n[2] Y . Zhang, Y . Li, L. Cui, D. Cai, L. Liu, T. Fu, X. Huang, E. Zhao,\nY . Zhang, Y . Chenet al., “Siren’s song in the ai ocean: A survey on hal-\nlucination in large language models,” arXiv preprint arXiv:2309.01219,\n2023.',
  'metadata': {'source': 'pdfs\\research2.pdf',
   'author': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'keywords': '',
   'trapped': '/False',
   'page': 16,
   'title': '',
   'creator': 'LaTeX with hyperref',
   'moddate': '2024-03-28T00:54:45+00:00',
   'total_pages': 21,
   'page_label': '17',
   'producer': 'pdfTeX-1.40.25',
   'content_length': 446,
   'creationdate': '2024-03-28T00:54:45+0

## Integrate with LLMs

In [35]:
API_KEY_GEMINI = "paste-your-api-key-here"

In [36]:
# !pip install langchain-google-genai

In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    google_api_key=API_KEY_GEMINI,
    model="gemini-2.5-flash",
    temperature=0.1,
    max_output_tokens=1024
)

In [38]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [41]:
answer = generate_output("what is Retrieval-Augmented Generation?", rag_retriever, llm)

Batches: 100%|██████████| 1/1 [00:00<00:00, 41.43it/s]


embeddings shape: (1, 384)
retrieved 3 documents


In [42]:
print(answer)

Based on the context, Retrieval-Augmented Generation (RAG) is a framework or system composed of three core components: Retrieval, Generation, and Augmentation.

*   **Retrieval** focuses on optimization methods including indexing, query, and embedding optimization.
*   **Generation** concentrates on the post-retrieval process and LLM fine-tuning.
*   **Augmentation** involves specific processes.

The context also discusses RAG's downstream tasks, evaluation system, and current challenges.
